# Imports

In [1]:
import pandas as pd
import igraph as ig
import numpy as np
import io
import matplotlib.pyplot as plt
from typing import Literal
import itertools
import seaborn as sns


In [2]:
def build_custom_network(properties_path, social_path, 
                         included_categories=None, 
                         included_activities=None, 
                         include_social=True, 
                         only_social=False):
    
    df_prop = pd.read_csv(properties_path)
    df_soc = pd.read_csv(social_path)    
    g = ig.Graph(directed=False)
    all_people_ids = [f"P_{id_val}" for id_val in df_prop['id'].unique()]
    g.add_vertices(all_people_ids)
    
    edges = []
    
    if not only_social:
        if included_categories:
            for cat in included_categories:
                for _, row in df_prop.iterrows():
                    person = f"P_{row['id']}"
                    prop_node = f"{cat.replace('_',' ').capitalize()}_{row[cat]}"    
                    if prop_node not in g.vs["name"]:
                        g.add_vertex(prop_node)
                    edges.append((person, prop_node))
        
        if included_activities:
            for act in included_activities:
                for _, row in df_prop.iterrows():
                    if row[act] == 1:
                        person = f"P_{row['id']}"
                        act_node = act.replace('_', ' ').title()
                        if act_node not in g.vs["name"]:
                            g.add_vertex(act_node)
                        edges.append((person, act_node))

    # 3. Handle Social Connections
    if include_social or only_social:
        for _, row in df_soc.iterrows():
            p1, p2 = f"P_{row['node_i']}", f"P_{row['node_j']}"
            # Only add social edge if both people exist in our properties df
            if p1 in g.vs["name"] and p2 in g.vs["name"]:
                edges.append((p1, p2))

    # 4. Add Edges to the graph
    g.add_edges(edges)

    # 5. Set Attributes
    g.vs["type"] = [1 if not n.startswith("P_") else 0 for n in g.vs["name"]]
    g.vs["color"] = ["#6495ED" if t == 0 else "#FFA500" for t in g.vs["type"]]
    
    # Handle edge styling
    edge_layers = []
    for e in g.es:
        s, t = g.vs[e.source]["name"], g.vs[e.target]["name"]
        if s.startswith("P_") and t.startswith("P_"):
            edge_layers.append("social")
        else:
            edge_layers.append("affiliation")
    
    g.es["color"] = ["#444444" if l == "social" else "#CCCCCC" for l in edge_layers]
    g.es["width"] = [1.5 if l == "social" else 0.5 for l in edge_layers]

    return g

# The plot_network function remains the same as your previous version
def plot_network(g, title="Network Visualization"):
    if g is None: return
    
    layout = g.layout_fruchterman_reingold()
    fig, ax = plt.subplots(figsize=(10, 10)) # Larger figure size helps
    ig.plot(
        g,
        target=ax,
        layout=layout,
        vertex_size=25,
        vertex_label=g.vs["name"],
        vertex_label_size=8,
        
        vertex_color=g.vs["color"],
        edge_width=g.es["width"],
        edge_color=g.es["color"],
        margin=100
    )
    plt.title(title)
    plt.show()





In [3]:
social_graph_arr = []
affiliation_graph_arr = []
social_affiliation_graph_arr = []
all_categories = ["gender", "age", "studies", "class_number"]
all_activities = ["plays_football", "watches_movies", "club", "smokes"]

days = [1, 30, 60, 90]

for i in days:
    prop_path = f"data/Part_B/properties_day_{i}.csv"
    conn_path = f"data/Part_B/connections_day_{i}.csv"    
    social_graph = build_custom_network(
        prop_path, conn_path,
        included_categories=[], 
        included_activities=[],
        include_social=True,
        only_social=True
    )
    deg = social_graph.degree()
    norm_deg = [d / (social_graph.vcount() - 1) for d in deg]
    social_graph.vs["degree_centrality"] = norm_deg
    social_graph_arr.append(social_graph)  

    affiliation_graph = build_custom_network(
        prop_path, conn_path,
        included_categories=all_categories,
        included_activities=all_activities,
        include_social=False,
        only_social=False
    )

    deg = affiliation_graph.degree()
    norm_deg = [d / (affiliation_graph.vcount() - 1) for d in deg]
    affiliation_graph.vs["degree_centrality"] = norm_deg
    affiliation_graph_arr.append(affiliation_graph)

    social_affiliation_graph = build_custom_network(
        prop_path, conn_path,
        included_categories=all_categories,
        included_activities=all_activities,
        include_social=True,
        only_social=False
    )
    deg = social_affiliation_graph.degree()
    norm_deg = [d / (social_affiliation_graph.vcount() - 1) for d in deg]
    social_affiliation_graph.vs["degree_centrality"] = norm_deg
    social_affiliation_graph_arr.append(social_affiliation_graph)

In [4]:
def loop_over_graph_triads(graph:ig.Graph):
    arr = []
    for i in range(len(graph.vs)):
        for j in range(i+1,len(graph.vs)):
            for k in range(j+1,len(graph.vs)):
                arr.append((i,j,k))
    return arr
def find_closed_triads(graph:ig.Graph):
    closed_triads = []
    for i,j,k in loop_over_graph_triads(graph):
        if (graph.are_adjacent(graph.vs[i],graph.vs[j]) and graph.are_adjacent(graph.vs[i],graph.vs[k])
                         and graph.are_adjacent(graph.vs[j],graph.vs[k])):
            closed_triads.append((i,j,k))
    return closed_triads
def find_open_triads(graph:ig.Graph,number_of_missing_edges = 1):
    open_triads = []
    # assert number_of_missing_edges!=0, "Use find_closed_triads function"
    for i,j,k in loop_over_graph_triads(graph):
        if (((not graph.are_adjacent(graph.vs[i],graph.vs[j])) + (not graph.are_adjacent(graph.vs[i],graph.vs[k]))
                    + (not graph.are_adjacent(graph.vs[j],graph.vs[k])))==number_of_missing_edges):
            open_triads.append((i,j,k))
    return open_triads

def find_intersection_count(open_triads:list[set[int]],closed_triads:list[set[int]]):
    intersaction_count = 0
    for i in open_triads:
        if i in closed_triads:
            intersaction_count+=1
    return intersaction_count


In [5]:
closed_triads:list[list[set[int]]] = []
open_triads:list[list[set[int]]] = []
for i,day in enumerate([1,30,60,90]):
    closed_triads.append(find_closed_triads(social_graph_arr[i]))
    open_triads.append(find_open_triads(social_graph_arr[i],1))
    print(30*"-")
    print(f"closed triads length at day: {day} :", len(closed_triads[i]))
    print(f"open triads missing 1 edge at day: {day} :",len(open_triads[i]))
    if i>=1:
        print("number of triadic closure: ",find_intersection_count(open_triads=open_triads[i-1],closed_triads=closed_triads[i]))



------------------------------
closed triads length at day: 1 : 0
open triads missing 1 edge at day: 1 : 12
------------------------------
closed triads length at day: 30 : 328
open triads missing 1 edge at day: 30 : 2081
number of triadic closure:  12
------------------------------
closed triads length at day: 60 : 5946
open triads missing 1 edge at day: 60 : 13779
number of triadic closure:  1424
------------------------------
closed triads length at day: 90 : 14812
open triads missing 1 edge at day: 90 : 21684
number of triadic closure:  2514


In [6]:
def get_attribute_vertices(g, attributes):
    formatted_attributes = [a.replace('_', ' ').title() for a in attributes]
    selected_vs = g.vs.select(lambda v: any(
        v["name"].startswith(attr.capitalize() + "_") or v["name"] == attr 
        for attr in formatted_attributes
    ))
    
    return selected_vs

def loop_over_graph_pairs(graph:ig.Graph,attributes_indexs):
    arr = []
    for i in range(len(graph.vs)):
        for j in range(i+1,len(graph.vs)):
            if i in attributes_indexs or j in attributes_indexs : continue
            arr.append((i,j))
    return arr


def find_closed_attribute_triads(graph:ig.Graph,attributes,attributes_indexs):
    closed_triads = []
    k= attributes.index
    for i,j in loop_over_graph_pairs(graph,attributes_indexs):
        if (graph.are_adjacent(graph.vs[i],graph.vs[j]) and graph.are_adjacent(graph.vs[i],graph.vs[k])
                        and graph.are_adjacent(graph.vs[j],graph.vs[k])):
            closed_triads.append((i,j,k))
    return closed_triads

def find_open_triads(graph:ig.Graph,attributes,attributes_indexs,mode:Literal["focal","membership"] = "focal"):
    open_triads = []
    k= attributes.index
    for i,j in loop_over_graph_pairs(graph,attributes_indexs):
        if mode == "focal":
            if (graph.are_adjacent(graph.vs[i],graph.vs[k]) and graph.are_adjacent(graph.vs[j],graph.vs[k])
                and not graph.are_adjacent(graph.vs[i],graph.vs[j])):
                open_triads.append((i,j))
        elif mode == "membership":
            if (graph.are_adjacent(graph.vs[i],graph.vs[j]) and 
                (graph.are_adjacent(graph.vs[j],graph.vs[k]) ^ graph.are_adjacent(graph.vs[i],graph.vs[k]))):
                open_triads.append((i,j))
        else:
            raise ValueError

    return open_triads


def find_intersection_count(open_triads_prev, closed_triads_curr):
    prev_pairs = set(tuple(sorted((t[0], t[1]))) for t in open_triads_prev)
    curr_pairs = set(tuple(sorted((t[0], t[1]))) for t in closed_triads_curr)
    return len(prev_pairs.intersection(curr_pairs))

def get_attribute_stats(g_curr, g_prev, attribute_name):
    try:
        attr_node_curr = g_curr.vs.find(name=attribute_name)
        current_people = set([n["name"] for n in attr_node_curr.neighbors()])
    except ValueError:
        current_people = set()
    new_count = 0
    left_count = 0
    
    if g_prev is not None:
        try:
            attr_node_prev = g_prev.vs.find(name=attribute_name)
            prev_people = set([n["name"] for n in attr_node_prev.neighbors()])                
            new_count = len(current_people - prev_people)            
            left_count = len(prev_people - current_people)
        except ValueError:            
            new_count = len(current_people)
            left_count = 0
    else:        
        new_count = len(current_people)
        left_count = 0
        
    return len(current_people), new_count, left_count

In [7]:
def get_shared_membership(g, attr_a, attr_b,attr_c = None):    
    try:    
        node_a = g.vs.find(name=attr_a)
        people_a = set(n["name"] for n in node_a.neighbors())

        node_b = g.vs.find(name=attr_b)
        people_b = set(n["name"] for n in node_b.neighbors())

        shared = people_a.intersection(people_b)
        if attr_c is not None : 
            node_c = g.vs.find(name=attr_c)
            people_c = set(n["name"] for n in node_c.neighbors())
            shared = people_c.intersection(shared)
        return len(shared), shared
    except ValueError:
        return 0, set()

import itertools

def get_count_friends_with_context(g, class_a, class_b, third_attr): 
    try:        
        people_a = set(n.index for n in g.vs.find(name=class_a).neighbors())
        people_b = set(n.index for n in g.vs.find(name=class_b).neighbors())                
        attr_node = g.vs.find(name=third_attr)
        people_with_attr = set(n.index for n in attr_node.neighbors())
        count = 0
        seen_edges = set()
        for i in people_a:
            for j in people_b:            
                if i != j and g.are_adjacent(i, j):                    
                    edge = tuple(sorted((i, j)))
                    if edge not in seen_edges:
                        if (i in people_with_attr or j in people_with_attr):
                            count += 1
                            seen_edges.add(edge)
        return count
    except ValueError:
        return 0

In [8]:
def print_shared_attribute_report(g, day_label):
    print(f"\n--- Shared Attributes Report: {day_label} ---")        
    nodes = [v["name"] for v in g.vs if not v["name"].startswith("P_")]    
    pairs = list(itertools.combinations(nodes, 2))
    print(f"{'Attribute A':<20} | {'Attribute B':<20} | {'Shared People'}")
    print("-" * 60)
    for a, b in pairs:
        count, members = get_shared_membership(g, a, b)
        if count > 0:
            print(f"{a:<20} | {b:<20} | {count}")

def plot_attribute_correlation(g):
    nodes = [v["name"] for v in g.vs if not v["name"].startswith("P_")]
    matrix = pd.DataFrame(0, index=nodes, columns=nodes)
    
    for a in nodes:
        for b in nodes:
            count, _ = get_shared_membership(g, a, b)
            matrix.loc[a, b] = count
            
    plt.figure(figsize=(12, 10))
    sns.heatmap(matrix, annot=True, cmap="YlGnBu")
    plt.title("Number of People Sharing Attributes")
    plt.show()
    
    
def sort_vs(g):
    lis = [i for i in g.vs if "P_" in i["name"]]
    return sorted(lis,key=lambda t: t["degree_centrality"],reverse=True)

def print_student_properties(g,student,current_properties):
    props = []
    for i in current_properties:
        if g.are_adjacent(student,i):
            props.append(True)
        else:
            props.append(False)
    return props

In [12]:

days = [1, 30, 60, 90]
all_categories = ["class_number"]
all_activities = ["smokes"]

all_categories = ["gender", "age", "studies", "class_number"]
all_activities = ["plays_football", "watches_movies", "club", "smokes"]


all_attr_names = all_activities + all_categories


# Storage for cross-day comparisons
closed_results = {}          
open_focal_results = {}      
open_membership_results = {} 

days = [1, 30, 60, 90]
all_attr_names = all_activities + all_categories

for day_idx, day in enumerate(days):
    g = social_affiliation_graph_arr[day_idx]
    g_prev = social_affiliation_graph_arr[day_idx-1] if day_idx > 0 else None
    
    # 1. Setup Property Nodes
    current_properties = get_attribute_vertices(g, all_attr_names)
    prop_indices = [p.index for p in current_properties]
    prop_names = [p["name"] for p in current_properties]
    
    print(f"\n{'='*30} DAY {day} {'='*30}")
    
    # 2. Part A: Individual Attribute Dynamics & Closures
    print(f"{'Attribute':<18} | {'Total':<5} | {'New':<3} | {'Left':<4} | {'Focal Cl.':<9} | {'Memb. Cl.'}")
    print("-" * 70)
    
    for prop_vertex in current_properties:
        p_name = prop_vertex["name"]
        total, new, left = get_attribute_stats(g, g_prev, p_name)
        
        closed = find_closed_attribute_triads(g, prop_vertex, prop_indices)
        # Store for day-to-day comparison logic
        closed_results.setdefault(day_idx, {})[p_name] = closed
        open_focal_results.setdefault(day_idx, {})[p_name] = find_open_triads(g, prop_vertex, prop_indices, mode="focal")
        open_membership_results.setdefault(day_idx, {})[p_name] = find_open_triads(g, prop_vertex, prop_indices, mode="membership")
        
        f_closures, m_closures = 0, 0
        if day_idx > 0 and p_name in open_focal_results[day_idx-1]:
            f_closures = find_intersection_count(open_focal_results[day_idx-1][p_name], closed)
            m_closures = find_intersection_count(open_membership_results[day_idx-1][p_name], closed)
        
        if total > 0 or left > 0:
            l_str = left if day_idx > 0 else "-"
            f_str = f_closures if day_idx > 0 else "-"
            m_str = m_closures if day_idx > 0 else "-"
            print(f"{p_name:<18} | {total:<5} | {new:<3} | {l_str:<4} | {f_str:<9} | {m_str}")

    # 3. Part B: Shared Attributes (Intersections)
    print(f"\nShared Attributes (Attribute Overlap):")
    shared_pairs = list(itertools.combinations(prop_names, 2))
    found_shared = False
    
    # We display pairs that share at least 2 people to keep the list meaningful
    for a, b in shared_pairs:
        shared_count,_ = get_shared_membership(g, a, b)
        if shared_count >= 1:
            print(f"  - {a} & {b} : {shared_count} shared people")
            found_shared = True
    
    if not found_shared:
        print("  No shared attributes found.")
        
    context_attribute = "Smokes" 
    print(f"\nSocial Ties in Classes involving '{context_attribute}':")
    class_nodes = [n for n in prop_names if "Class" in n]
    class_pairs = list(itertools.combinations_with_replacement(class_nodes, 2))
    for a, b in class_pairs:
        num_context_friends = get_count_friends_with_context(g, a, b, context_attribute)
        
        if num_context_friends > 0:
            label = f"Within {a}" if a == b else f"Between {a} & {b}"
            print(f"  - {label} : {num_context_friends} friendships involve at least one {context_attribute}")  


    print(f"\nTop 5 Students in terms of Degree Centrality:")
    print(F"\n{"Id":<5} | ",end="")
    for attr in current_properties:
        print(f"{attr["name"]:<18} | ",end="")
    print()
    print("-" * 180,end="")
    for student in sort_vs(g)[:5]:
        props = print_student_properties(g,student,current_properties)
        print(F"\n{student.index:<5} | ",end="")
        for prop in props:
            print(f"{prop:<18} | ",end="")
        print(f"{round(student["degree_centrality"],3):<19} | ",end="")
        
    print()
    print("-" * 80)


============================== DAY 1 ==============================
Attribute          | Total | New | Left | Focal Cl. | Memb. Cl.
----------------------------------------------------------------------
Gender_boy         | 87    | 87  | -    | -         | -
Gender_girl        | 33    | 33  | -    | -         | -
Age_17             | 29    | 29  | -    | -         | -
Age_18             | 22    | 22  | -    | -         | -
Age_16             | 22    | 22  | -    | -         | -
Age_15             | 24    | 24  | -    | -         | -
Age_14             | 23    | 23  | -    | -         | -
Studies_3          | 28    | 28  | -    | -         | -
Studies_4          | 18    | 18  | -    | -         | -
Studies_1          | 34    | 34  | -    | -         | -
Studies_2          | 27    | 27  | -    | -         | -
Studies_5          | 13    | 13  | -    | -         | -
Class number_1     | 30    | 30  | -    | -         | -
Class number_2     | 30    | 30  | -    | -         | -
Class number

In [10]:
def compare_smoker_profiles(g):
    smokers = g.vs.select(lambda v: v["name"].startswith("P_") and 
                          any(n["name"] == "Smokes" for n in v.neighbors()))
    
    non_smokers = g.vs.select(lambda v: v["name"].startswith("P_") and 
                              not any(n["name"] == "Smokes" for n in v.neighbors()))
    
    stats = []
    for group, name in [(smokers, "Smoker"), (non_smokers, "Non-Smoker")]:
        avg_age = sum([int(v["age"]) for v in group]) / len(group) if len(group) > 0 else 0
        avg_studies = sum([int(v["studies"]) for v in group]) / len(group) if len(group) > 0 else 0
        avg_degree = sum(v.degree() for v in group) / len(group) if len(group) > 0 else 0
        stats.append({"Group": name, "Count": len(group), "Age": avg_age, "Studies": avg_studies, "Social Ties": avg_degree})
        
    return pd.DataFrame(stats)

In [11]:
evolution_data = []

for day_idx, day in enumerate([1, 30, 60, 90]):
    g = social_affiliation_graph_arr[day_idx]
    
    people = g.vs.select(lambda v: v["name"].startswith("P_"))
    
    for p in people:
        is_smoker = any(n["name"] == "Smokes" for n in p.neighbors())
        plays_football = any(n["name"] == "Plays Football" for n in p.neighbors())
        watches_movies = any(n["name"] == "Watches Movies" for n in p.neighbors())
        club = any(n["name"] == "Club" for n in p.neighbors())
        study = 0
        # Social Ties (counting only edges to other people)
        for i in range(1,6):
            if any(n["name"] == f"Studies_{i}" for n in p.neighbors()):
                study= i 
                break
        age = 0 
        for i in range(14,19):
            if any(n["name"] == f"Age_{i}" for n in p.neighbors()):
                age = i 
                break
        gender = -1
        for idx,i in enumerate(["boy","girl"]):
            if any(n["name"] == f"Gender_{i}" for n in p.neighbors()):
                gender = idx 
                break

        assert age!=0
        assert study!=0
        assert gender!=-1

        

        friends = [n for n in p.neighbors() if n["name"].startswith("P_")]
        
        evolution_data.append({
            "Day": day,
            "Is Smoker": is_smoker,
            "Social Connections": len(friends),
            "Football": plays_football,
            "Movies": watches_movies,
            "Clubs": club,
            "Study":study,
            "Age":age,
            "Gender":gender,
        })

df_evo = pd.DataFrame(evolution_data)

summary = df_evo.groupby(["Day", "Is Smoker"]).agg({
    "Social Connections": "mean",
    "Football": "mean",
    "Movies": "mean",
    "Clubs" : "mean",
    "Study":"mean",
    "Age":"mean",
    "Gender":"mean"
}).reset_index()

print(summary)

   Day  Is Smoker  Social Connections  Football    Movies     Clubs     Study  \
0    1      False            0.440000  0.250000  0.430000  0.200000  2.570000   
1    1       True            0.300000  0.550000  0.250000  0.200000  2.600000   
2   30      False            2.450549  0.318681  0.428571  0.241758  2.527473   
3   30       True           13.827586  0.551724  0.586207  0.172414  2.620690   
4   60      False            8.500000  0.487179  0.602564  0.448718  2.564103   
5   60       True           32.880952  0.595238  0.595238  0.214286  1.523810   
6   90      False           21.140000  0.540000  0.660000  0.680000  2.060000   
7   90       True           37.614286  0.714286  0.757143  0.314286  1.085714   

         Age    Gender  
0  15.860000  0.310000  
1  16.850000  0.100000  
2  15.802198  0.307692  
3  16.724138  0.172414  
4  15.794872  0.282051  
5  16.452381  0.261905  
6  15.900000  0.260000  
7  16.114286  0.285714  
